In [2]:
import numpy as np
import pandas as pd
import torch

from equipment_forecasting.test_a_data import (
    HISTORY_COLUMNS,
    load_hourly_equipment,
)
from load_forecasting.checkpoint import load_checkpoint
from load_forecasting.data import (
    _interpolate_short_gaps,
    future_weather_frame,
    wavelet_feature_window,
)
from load_forecasting.predict import restore_model
from train_equipment_a_july16 import predict_equipment

data_path = (
    "附件4：测试数据集/"
    "测试数据项目A历史数据_2025年4月、7月、9月.xlsx"
)

load_checkpoint_data = load_checkpoint(
    "outputs/load/best.pt", "cpu"
)
equipment_checkpoint = load_checkpoint(
    "outputs/equipment/best.pt", "cpu"
)

load_model = restore_model(load_checkpoint_data)
hourly = load_hourly_equipment(data_path).set_index("timeStamp")

start = pd.Timestamp("2025-07-15 00:00:00")
end = pd.Timestamp("2025-07-17 00:00:00")

results = []

for target_time in pd.date_range(
    start, end, freq="h", inclusive="left"
):
    history_times = pd.date_range(
        target_time - pd.Timedelta(hours=24),
        periods=24,
        freq="h",
    )

    history = hourly.reindex(history_times).reset_index()
    history = history.rename(columns={"index": "timeStamp"})

    history[HISTORY_COLUMNS] = _interpolate_short_gaps(
        history[HISTORY_COLUMNS],
        limit=3,
    )

    if history[HISTORY_COLUMNS].isna().any().any():
        print("跳过，历史不完整：", target_time)
        continue

    features, feature_names = wavelet_feature_window(
        history,
        HISTORY_COLUMNS,
        level=1,
        wavelet="db4",
    )

    future = hourly.loc[[target_time]].reset_index()
    weather, weather_names = future_weather_frame(future)

    history_x = features.to_numpy(
        dtype=np.float32
    )[None, ...]

    future_weather = weather.to_numpy(
        dtype=np.float32
    )[None, ...]

    # 冷负荷预测
    x_norm = (
        history_x - load_checkpoint_data["x_mean"]
    ) / load_checkpoint_data["x_std"]

    w_norm = (
        future_weather
        - load_checkpoint_data["future_weather_mean"]
    ) / load_checkpoint_data["future_weather_std"]

    with torch.no_grad():
        load_prediction = load_model(
            torch.from_numpy(x_norm.astype(np.float32)),
            torch.from_numpy(w_norm.astype(np.float32)),
        ).numpy()[0, 0]

    load_prediction = (
        load_prediction * load_checkpoint_data["y_std"]
        + load_checkpoint_data["y_mean"]
    )

    # 27项设备参数预测
    equipment_prediction = predict_equipment(
        equipment_checkpoint,
        history_x,
        future_weather,
        device="cpu",
    )[0, 0]

    row = {
        "timeStamp": target_time,
        "predicted_load": load_prediction,
    }

    for name, value in zip(
        equipment_checkpoint["target_names"],
        equipment_prediction,
    ):
        row[name] = value

    results.append(row)

result = pd.DataFrame(results)
result.to_csv(
    "hourly_prediction.csv",
    index=False,
    encoding="utf-8-sig",
)

print(result)

ModuleNotFoundError: No module named 'numpy._core'